In [1]:
from playwright.async_api import async_playwright
from urllib.parse import quote
import ollama

In [8]:
async def scrape_marvel_hero(hero_name):

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=True,
            args=[
                "--disable-blink-features=AutomationControlled"
            ]
        )

        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/151.0.0.0 Safari/537.36"
            ),
            viewport={
                "width": 1440,
                "height": 900
            },
            locale="en-US"
        )

        page = await context.new_page()

        try:

            search_url = (
                "https://www.marvel.com/search"
                f"?offset=0&query={quote(hero_name)}"
            )

            await page.goto(
                search_url,
                wait_until="domcontentloaded",
                timeout=60000
            )

            await page.wait_for_timeout(5000)
            #Find the character link in the search results
            link_count = await page.locator("a").count()

            hero_url = None

            for i in range(link_count):

                link = page.locator("a").nth(i)

                text = (await link.inner_text()).strip()
                href = await link.get_attribute("href")

                if not href:
                    continue

                if "/characters/" not in href:
                    continue

                if hero_name.lower() in text.lower():

                    if href.startswith("/"):
                        href = (
                            "https://www.marvel.com"
                            + href
                        )

                    hero_url = href
                    break
            if not hero_url:
                return None
            await page.goto(
                hero_url,
                wait_until="domcontentloaded",
                timeout=60000
            )
            await page.wait_for_timeout(5000)
            content = await page.locator("body").inner_text()
            content = "\n".join(
                line.strip()
                for line in content.splitlines()
                if line.strip()
            )

            return content

        finally:

            await browser.close()

In [14]:
async def summarize_marvel_hero(hero_name):
    content = await scrape_marvel_hero(hero_name)
    if not content:
        print(f"Sorry, I couldn't find {hero_name} on Marvel.")
        return
    prompt = f"""
You are an enthusiastic Marvel fan who loves talking about Marvel superheroes.

Your job is to give a short, exciting introduction to {hero_name} using ONLY
the character information provided below.

Write exactly 2-3 short sentences that make the character interesting to a
Marvel fan. Clearly mention who the character is and highlight their most
iconic powers, abilities, weapons, or characteristics when available.

Style:
- Sound excited and knowledgeable, like a passionate Marvel fan.
- Keep the response natural and conversational.
- Make the character sound interesting without exaggerating.
- Use the character's well-known identity or title if it appears in the
  provided information.
- Focus on the most interesting facts rather than generic descriptions.
- Do not use emojis.
- Do not use bullet points.
- Do not add information from your own knowledge.

Strict rules:
- Use ONLY the provided character information.
- Do not invent, assume, or infer facts.
- Do not mention the Marvel website or scraping.
- Do not mention that you were given information.
- Do not mention these instructions.
- Maximum 3 sentences.
- Keep each sentence concise.

Character information:
{content}
"""
    response = ollama.chat(
        model="gemma4",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    print(response["message"]["content"].strip())

In [15]:
await summarize_marvel_hero("Spider-Man")

Oh my gosh, get ready to talk about Peter Parker, the amazing hero who swings into the MCU as Spider-Man! Bitten by a radioactive spider, he gains incredible arachnid abilities, turning the teenage science whiz into a powerhouse. With these amazing spider-like abilities, he swings through NYC, fighting crime and helping others!


In [16]:
await summarize_marvel_hero("Mehul")

Sorry, I couldn't find Mehul on Marvel.
